# Training dan Testing

In [21]:
!pip install firebase-admin scikit-learn pandas joblib xgboost

In [22]:
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, r2_score
import joblib

print("1. Memuat Dataset Terpisah (Train & Test)...")
# Sesuaikan path dengan lokasi dataset di Kaggle Anda
train_df = pd.read_csv('/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv')
print("Training Dataset: \n",train_df.head(10),"\n")
print("Testing Dataset: \n",test_df.head(10))

1. Memuat Dataset Terpisah (Train & Test)...
Training Dataset: 
    Id  MSSubClass MSZoning  LotFrontage  LotArea Street Alley LotShape  \
0   1          60       RL         65.0     8450   Pave   NaN      Reg   
1   2          20       RL         80.0     9600   Pave   NaN      Reg   
2   3          60       RL         68.0    11250   Pave   NaN      IR1   
3   4          70       RL         60.0     9550   Pave   NaN      IR1   
4   5          60       RL         84.0    14260   Pave   NaN      IR1   
5   6          50       RL         85.0    14115   Pave   NaN      IR1   
6   7          20       RL         75.0    10084   Pave   NaN      Reg   
7   8          60       RL          NaN    10382   Pave   NaN      IR1   
8   9          50       RM         51.0     6120   Pave   NaN      Reg   
9  10         190       RL         50.0     7420   Pave   NaN      Reg   

  LandContour Utilities  ... PoolArea PoolQC  Fence MiscFeature MiscVal  \
0         Lvl    AllPub  ...        0    NaN 

In [23]:
print("2. Membersihkan & Menyiapkan Data...")
features = ['GrLivArea', 'BedroomAbvGr', 'FullBath', 'GarageCars']

# Pisahkan Fitur (X) dan Target (y) dari data Train
X_train_full = train_df[features].fillna(0) # Mengisi nilai kosong dengan 0
y_train_full = train_df['SalePrice']

# Siapkan data Test (Data buta tanpa SalePrice)
X_test_unseen = test_df[features].fillna(0)

# Split data train untuk validasi internal (mencari nilai akurasi untuk Laporan)
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.2, random_state=42)

2. Membersihkan & Menyiapkan Data...


In [24]:
xgb_model = xgb.XGBRegressor(random_state=42)

param_grid = {
    'n_estimators': [100, 300, 500],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5, 7],
    'subsample': [0.8, 0.9, 1.0],
    'colsample_bytree': [0.8, 0.9, 1.0],
    'min_child_weight': [1, 3, 5]
}

random_search = RandomizedSearchCV(
    estimator=xgb_model, 
    param_distributions=param_grid, 
    n_iter=50,         
    scoring='r2',       
    cv=3,              
    verbose=1, 
    random_state=42,
    n_jobs=-1           
)

random_search.fit(X_train, y_train)

best_xgb_model = random_search.best_estimator_

print(f"Tuning Selesai! Parameter terbaik yang ditemukan:")
print(random_search.best_params_)

Fitting 3 folds for each of 50 candidates, totalling 150 fits
Tuning Selesai! Parameter terbaik yang ditemukan:
{'subsample': 0.8, 'n_estimators': 100, 'min_child_weight': 1, 'max_depth': 3, 'learning_rate': 0.1, 'colsample_bytree': 1.0}


In [25]:
print("\n4. Evaluasi Model")
y_pred_val = best_xgb_model.predict(X_val)
mae = mean_absolute_error(y_val, y_pred_val)
r2 = r2_score(y_val, y_pred_val)
print(f"   - Mean Absolute Error (MAE): ${mae:,.2f}")
print(f"   - R-squared (R2 Score): {r2:.4f}")

print("\n5. Mengekspor Model ke File untuk Microservice...")
joblib.dump(best_xgb_model, 'xgboost_harga_rumah_tuned.pkl')
print("Selesai! Model diekspor sebagai 'xgboost_harga_rumah_tuned.pkl'")


4. Evaluasi Model
   - Mean Absolute Error (MAE): $26,313.90
   - R-squared (R2 Score): 0.8231

5. Mengekspor Model ke File untuk Microservice...
Selesai! Model diekspor sebagai 'xgboost_harga_rumah_tuned.pkl'


# Running Model

In [26]:
import firebase_admin
from firebase_admin import credentials, db
import time
import joblib
import pandas as pd
import xgboost as xgb

In [27]:
print("Memuat model XGBoost dari penyimpanan...")
loaded_model = joblib.load('xgboost_harga_rumah_tuned.pkl')
features = ['GrLivArea', 'BedroomAbvGr', 'FullBath', 'GarageCars']

Memuat model XGBoost dari penyimpanan...


In [28]:
path_json = '/kaggle/input/datasets/nathanaelindra/firebaseq/cloud-dasboard-firebase-adminsdk-fbsvc-cecf087ccd.json' 
url_firebase = 'https://cloud-dasboard-default-rtdb.asia-southeast1.firebasedatabase.app/'  

try:
    app = firebase_admin.get_app()
except ValueError:
    cred = credentials.Certificate(path_json)
    firebase_admin.initialize_app(cred, {'databaseURL': url_firebase})

print("📡 AI XGBoost Standby. Terhubung ke Firebase. Menunggu request...")

📡 AI XGBoost Standby. Terhubung ke Firebase. Menunggu request...


In [29]:
while True:
    ref = db.reference('prediksi_harga')
    data = ref.get()
    
    if data:
        for key, val in data.items():
            if val.get('status') == 'pending':
                print(f"⬇️ Menerima request dari Web: {key}")
                
                # Mengambil data dari web
                luas = float(val.get('luas'))
                kamar = float(val.get('kamar'))
                mandi = float(val.get('mandi'))
                garasi = float(val.get('garasi'))
                
                # Memprediksi menggunakan XGBoost
                input_df = pd.DataFrame([[luas, kamar, mandi, garasi]], columns=features)
                estimasi = loaded_model.predict(input_df)[0]
                
                # Mengirim hasil ke Web
                ref.child(key).update({
                    'status': 'done',
                    'hasil_prediksi': f"Rp {int(estimasi * 15000):,}" # Asumsi USD to IDR
                })
                print(f"⬆️ Hasil XGBoost dikirim: Rp {int(estimasi * 15000):,}")
    
    time.sleep(1.5) # Jeda ringan agar stabil

KeyboardInterrupt: 